# A real `rclpy` node, running in this notebook

This is genuinely compiled `rclpy` — the same ROS 2 rolling Python client library used on real robots — running via [`xeus-python`](https://github.com/jupyter-xeus/xeus-python) on [`jupyterlite-xeus`](https://github.com/jupyter-xeus/jupyterlite-xeus), talking to ROS 2 over [`rmw_zenoh_pico`](https://github.com/esol-community/rmw_zenoh_pico). Not a mock, not a REST call dressed up to look like `rclpy` — the exact same `rclpy`/`rmw_zenoh_pico` build used by the [standalone rclpy talker demo](../index.html#demos), just running inside a notebook kernel instead of a plain web page, so you can edit and re-run cells to try things out.

**You need a zenoh router reachable at `ws://127.0.0.1:7447` for anything below to actually publish/receive** — see the main page's setup box for the one-line `pixi exec` command. Without one, `rclpy.init()`/node creation still work, but publishing will fail (matches the standalone demos' behavior, [documented limitation](../index.html#limits): the connect address is compiled in, not runtime-configurable, so this notebook can't point at a different router either).

In [ ]:
import os

# Same workarounds the standalone rclpy demo needs (see ../browser_demo/rclpy_boot.c
# and talker_rclpy.py) -- rcl_logging's default backend isn't wired up on this
# platform, and rmw_zenoh_pico is the only RMW built into this environment anyway.
os.environ['RCL_LOGGING_IMPLEMENTATION'] = 'rcl_logging_noop'
os.environ['RMW_IMPLEMENTATION'] = 'rmw_zenoh_pico'

import rclpy
from rclpy.node import Node
from rclpy.event_handler import PublisherEventCallbacks, SubscriptionEventCallbacks
from std_msgs.msg import String

rclpy.init(args=[])

# enable_rosout=False, start_parameter_services=False: sidesteps two more
# rclpy features that would otherwise pull in the same typesupport gap this
# whole pipeline works around elsewhere -- see ../docs/demo_env.md.
node = Node('wasm_rclpy_jupyter', enable_rosout=False, start_parameter_services=False)
print('Node created:', node.get_name())

## Publisher and subscriber

Both on `chatter_jupyter` — publish from here and you'll see it echoed back below, or open [the rclc talker demo](../demos/index_rclc.html) or [the rclpy talker demo](../index.html#demos) in another tab and watch its own topic independently, or point a native `ros2 topic echo /chatter_jupyter` at the same router to see this notebook's own messages arrive outside the browser entirely.

`use_default_callbacks=False`: `rmw_zenoh_pico` doesn't support QoS event callbacks yet (the same [known limitation](../index.html#limits) the compiled demos work around) — leaving the defaults on raises instead of silently no-oping.

In [ ]:
received = []

def _on_message(msg):
    received.append(msg.data)
    print('received:', msg.data)

pub = node.create_publisher(
    String, 'chatter_jupyter', 10,
    event_callbacks=PublisherEventCallbacks(use_default_callbacks=False))

sub = node.create_subscription(
    String, 'chatter_jupyter', _on_message, 10,
    event_callbacks=SubscriptionEventCallbacks(use_default_callbacks=False))

print('Publisher + subscriber ready on chatter_jupyter')

## Play around

Edit the string below and re-run this cell as many times as you like — each run publishes one message, then spins the node briefly so any reply (including your own, echoed back by the subscriber above) has a chance to arrive before the cell finishes.

In [ ]:
msg = String()
msg.data = 'hello from the notebook!'
pub.publish(msg)
print('published:', msg.data)

for _ in range(20):
    rclpy.spin_once(node, timeout_sec=0.1)

## Cleanup

Run this once you're done — same as any real `rclpy` script — before re-running the setup cell above from scratch (re-running it without cleaning up first will error, since the node already exists).

In [ ]:
node.destroy_node()
rclpy.shutdown()
print('shut down cleanly')